<a href="https://colab.research.google.com/github/shubham-dg10/AutomateChatGPTpromptswithPython/blob/main/stable/f222_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, requests, subprocess, time, re, atexit
from random import randint
from threading import Timer
from queue import Queue

# --- 1. CLEAN START ---
print('Step 1: Cleaning environment...')
%cd /content
!rm -rf /content/*

# --- 2. INSTALL COMFYUI & DEPENDENCIES ---
print('Step 2: Installing ComfyUI...')
!apt -y update -qq && !apt -y install -qq aria2
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q torch torchvision torchaudio xformers mediapipe InsightFace piexif blake3 dynamicprompts ultralytics segment-anything onnxruntime-gpu

# --- 3. INSTALL CUSTOM NODES ---
print('Step 3: Installing Custom Nodes...')
%cd /content/ComfyUI/custom_nodes
nodes = [
    'https://github.com/ltdrdata/ComfyUI-Manager',
    'https://github.com/cubiq/ComfyUI_IPAdapter_plus',
    'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'https://github.com/cubiq/ComfyUI_Essentials',
    'https://github.com/Fannovel16/comfyui_controlnet_aux'
]
for repo in nodes:
    !git clone {repo}

# --- 4. DOWNLOAD MODELS ---
print('Step 4: Downloading Models...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M 'https://civitai.com/api/download/models/348913' -d /content/ComfyUI/models/checkpoints -o juggernautXL_v9.safetensors
!mkdir -p /content/ComfyUI/models/ipadapter /content/ComfyUI/models/clip_vision
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M 'https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin' -d /content/ComfyUI/models/ipadapter -o ip-adapter-faceid-plusv2_sdxl.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M 'https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors' -d /content/ComfyUI/models/clip_vision -o CLIP-ViT-H-fp16.safetensors

# --- 5. SETUP ACCESS URL ---
print('Step 5: Starting Tunnel...')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod 777 /content/cloudflared

def run_tunnel(port, metrics_port, q):
    p = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    atexit.register(p.terminate)
    for _ in range(15):
        time.sleep(2)
        try:
            res = requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text
            url = re.search('(?P<url>https?://[^\\s]+.trycloudflare.com)', res).group('url')
            if url:
                q.put(url)
                return
        except: pass

out_q = Queue()
Timer(2, run_tunnel, args=(8188, randint(8100, 9000), out_q)).start()
tunnel_url = out_q.get()

# --- 6. LAUNCH ---
%cd /content/ComfyUI
print(f'\n--- READY ---\nURL: {tunnel_url}\n--------------')
!python main.py --dont-print-server